# Featured ATL → OMA flight: overflow response versus atmospheric depth

This notebook is the reproducible single-flight case study for the flight used in the featured animation. It treats the overflow channel as a detector-response observable and tests whether its live-time-normalized rate is better described by a linear, quadratic, zero-floor exponential, or floor-plus-exponential response to atmospheric column depth.

## tl;dr

The 16-bin ATL → OMA series contains 5,290 overflow counts in 6,590 s. A Poisson count model with live time as the exposure finds that the floor-plus-exponential response has the lowest deviance (18.64) and AICc. Its fitted floor is 0.0182 CPS (approximate 95% interval 0.0079–0.0422) and its effective depth scale is 151 g cm⁻² (136.8–166.9). The high-depth tail contains 25 counts in 721 s, or 0.0347 CPS with an exact 95% Poisson interval of 0.0224–0.0512 CPS. This is a one-flight descriptive result, not a direct cosmic-ray flux law or a universal attenuation length.

## Context & Methods

The source is the compact, per-flight pressure table in `data/derived/pressure_overflow_by_flight.csv`, derived from the raw RCSPG and route files listed in the repository manifest. The featured flight is identified as `OMA_ATL-OMA`.

The pressure/depth coordinates derive from FlightAware website data. They are included for noncommercial academic review but excluded from the repository's MIT and CC BY 4.0 licenses; see `THIRD_PARTY_NOTICES.md`.

For bin i, the model is `nᵢ ~ Poisson(Tᵢ R(Xᵢ))`, where `nᵢ` is overflow counts, `Tᵢ` is detector live time, and `R(Xᵢ)` is overflow CPS. Atmospheric depth X is the physical predictor. Pressure P is retained as a display coordinate because the current conversion is `X [g cm⁻²] = 1.019716 × P [hPa]`. The fitted response families are:

- `R = b₀ + b₁(X/1000)`
- `R = b₀ + b₁(X/1000) + b₂(X/1000)²`
- `R = A exp(-X/Λ)`
- `R = B + A exp(-X/Λ)`

The linear and quadratic curves use a non-negative identity-link Poisson fit; the exponential scale is profiled by one-dimensional likelihood search. Model comparison uses Poisson deviance, Pearson residuals, and small-sample corrected AIC (AICc). Absolute AIC values omit a shared constant, so only within-dataset differences are interpreted.

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd()
if not (REPO / 'scripts').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.analyze_featured_flight_overflow import run_analysis

result = run_analysis(
    REPO / 'data' / 'derived' / 'pressure_overflow_by_flight.csv',
    REPO / 'data' / 'derived',
    REPO / 'figures' / 'presentation' / 'figure_06_oma_atl_overflow_depth_response.png',
    'OMA_ATL-OMA',
)
data = result['data']
print(f"Loaded {result['summary']['flight_key']}: {len(data)} bins, {data.live_time_s.sum():.0f} s live time, {data.overflow_counts.sum():.0f} overflow counts")

Loaded OMA_ATL-OMA: 16 bins, 6590 s live time, 5290 overflow counts


## Data

Each row is a pressure bin. Rates are never fitted from raw counts alone: the live-time exposure is part of the likelihood. The plotted error bars are exact 95% Garwood Poisson intervals; the green band is an approximate 95% likelihood-curvature interval for the fitted mean response.

In [2]:
assert data.flight_key.nunique() == 1
assert (data.live_time_s > 0).all()
assert (data.overflow_counts >= 0).all()
ratio = data.column_depth_center_g_cm2 / data.pressure_center_hpa
assert (ratio - 1.019716).abs().max() < 1e-6
print('checks passed: positive live time; non-negative counts; X/P = 1.019716 for every bin')

checks passed: positive live time; non-negative counts; X/P = 1.019716 for every bin


In [3]:
models = result['model_table'].copy()
print(models[['model', 'delta_aicc', 'deviance', 'deviance_df', 'pearson_rmse', 'rate_rmse_cps']].rename(columns={'delta_aicc': 'delta_AICc', 'pearson_rmse': 'pearson_RMSE', 'rate_rmse_cps': 'rate_RMSE_CPS'}).to_string(index=False, formatters={'delta_AICc': '{:.3f}'.format, 'deviance': '{:.3f}'.format, 'pearson_RMSE': '{:.3f}'.format, 'rate_RMSE_CPS': '{:.4f}'.format}))

                 model  delta_AICc  deviance  deviance_df  pearson_RMSE  rate_RMSE_CPS
                linear     535.140   556.856           14         5.261         0.2926
             quadratic      65.192    83.830           13         2.204         0.1005
exponential_zero_floor       3.313    25.028           14         1.288         0.0416
exponential_plus_floor       0.000    18.639           13         1.061         0.0366


## Results

The floor-plus-exponential is the best of the four requested descriptions for this flight. Its fitted parameters are `B = 0.0182 CPS` (0.0079–0.0422), `A = 6.207 CPS` (5.210–7.394), and `Λ = 151.1 g cm⁻²` (136.8–166.9); intervals are approximate 95% Hessian-based likelihood intervals. The zero-floor exponential is close in AICc (ΔAICc = 3.31), so the floor is suggestive rather than decisively established by this single flight. The linear and quadratic descriptions leave much larger structured residuals.

In [4]:
tail = result['summary']['high_depth_tail']
print(f"high-depth tail: P >= {tail['threshold_pressure_hpa']:.0f} hPa; X >= {tail['threshold_depth_g_cm2']:.1f} g cm^-2; {tail['counts']:.0f} counts / {tail['live_time_s']:.0f} s = {tail['rate_cps']:.4f} CPS; exact 95% CI {tail['exact_95_low_cps']:.4f}-{tail['exact_95_high_cps']:.4f} CPS")
print(f"floor-plus-exponential tail prediction: {tail['exponential_plus_floor_exposure_weighted_rate_cps']:.4f} CPS")
print(f"zero-floor exponential tail prediction: {tail['exponential_zero_floor_exposure_weighted_rate_cps']:.4f} CPS")

high-depth tail: P >= 750 hPa; X >= 764.8 g cm^-2; 25 counts / 721 s = 0.0347 CPS; exact 95% CI 0.0224-0.0512 CPS
floor-plus-exponential tail prediction: 0.0348 CPS
zero-floor exponential tail prediction: 0.0246 CPS


In [5]:
from PIL import Image
figure = Image.open(REPO / 'figures' / 'presentation' / 'figure_06_oma_atl_overflow_depth_response.png')
print(f'Figure available at figures/presentation/figure_06_oma_atl_overflow_depth_response.png ({figure.size[0]} x {figure.size[1]} pixels)')

Figure available at figures/presentation/figure_06_oma_atl_overflow_depth_response.png (1800 x 1100 pixels)


## Takeaways

1. The overflow response is strongly depth-dependent over this flight, but the dependence is not well represented by a straight line.
2. A simple exponential captures most of the falloff; adding a non-negative floor improves the fit and matches the aggregated high-depth tail.
3. The floor is a detector-level response term. It could contain residual atmospheric secondaries, aircraft/material contributions, geometry, electronics/overflow behavior, or other backgrounds. It is not a measurement of a direct cosmic-ray flux.
4. The next scientific test is replication: fit the same count/exposure model to every flight, then use flight-specific intercepts or a hierarchical model before pooling routes, devices, shielding states, ocean voyages, and lead-castle measurements.